# BƯỚC 1: KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EXPLORATORY DATA ANALYSIS - EDA)

## Mục tiêu theo đề bài Practice Exercise 3:
1. Đọc và kiểm tra cấu trúc tập dữ liệu khách hàng (`data/raw/marketing_campaign.csv`, định dạng phân tách bằng tab `\t`).
2. Tạo các đặc trưng phân cụm cốt lõi (`Age = 2014 - Year_Birth`, `Total_Spending = sum(6 cột Mnt*)`), lọc lỗi miền bất hợp lý (`Year_Birth < 1920`, `Income > 200,000`) và chia 80% Train / 20% Test trước khi điền trung vị `Income`.
3. Phân tích giá trị khuyết thiếu (Missing values), phát hiện dị biệt (Outliers) bằng phương pháp IQR (Tukey's Fences).
4. Khảo sát hình thái phân phối, quan hệ song biến và biến nhân khẩu học thật (`Education`, `Marital_Status`).

In [1]:
# Thêm đường dẫn gốc của dự án để import module từ src/
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import NUMERICAL_FEATURES, PROCESSED_DATA_DIR, RAW_DATA_PATH
from src.data.loader import load_raw_data, split_data, validate_schema
from src.data.preprocessing import (
    add_derived_features,
    audit_missing_and_outliers,
    clean_data,
    compute_train_medians,
    filter_invalid_records,
)
from src.evaluation.visualization import plot_eda_correlations, plot_eda_distributions

In [2]:
# Nạp dữ liệu gốc và kiểm định tính toàn vẹn schema
raw_df = load_raw_data(RAW_DATA_PATH)
schema_info = validate_schema(raw_df)
raw_df.head()

 [NẠP DỮ LIỆU] Đã đọc thành công 2,240 dòng x 29 cột từ marketing_campaign.csv
 [KIỂM ĐỊNH SCHEMA] Trạng thái: HỢP LỆ | Dòng: 2,240 | Cột thiếu: 0 | ID trùng: 0 | ID thiếu: 0 | Giá trị khuyết: 24


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [3]:
# Kiểm toán giá trị khuyết, ngoại lai (IQR) và dị biệt hạng mục trên dữ liệu gốc trước khi làm sạch
raw_audit_df = audit_missing_and_outliers(raw_df)
print("Giá trị Marital_Status thô:", raw_df["Marital_Status"].value_counts().to_dict())
print("Năm sinh < 1920:", raw_df[raw_df["Year_Birth"] < 1920][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
print("Thu nhập > 200,000:", raw_df[raw_df["Income"] > 200000][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
raw_audit_df

 [KIỂM TOÁN DỮ LIỆU] Đã hoàn tất thống kê Null & IQR Outliers cho 3 đặc trưng.
Giá trị Marital_Status thô: {'Married': 864, 'Together': 580, 'Single': 480, 'Divorced': 232, 'Widow': 77, 'Alone': 3, 'Absurd': 2, 'YOLO': 2}
Năm sinh < 1920: [{'ID': 7829, 'Year_Birth': 1900, 'Income': 36640.0}, {'ID': 11004, 'Year_Birth': 1893, 'Income': 60182.0}, {'ID': 1150, 'Year_Birth': 1899, 'Income': 83532.0}]
Thu nhập > 200,000: [{'ID': 9432, 'Year_Birth': 1977, 'Income': 666666.0}]


,Đặc trưng,Null Count,Null %,Q1,Q3,IQR,Ngưỡng dưới,Ngưỡng trên,Số lượng Outlier,Tỷ lệ Outlier %
0,Age,0,0.00,37.00,55.0,18.00,10.00,82.00,3,0.13
1,Income,24,1.07,35303.00,68522.0,33219.00,-14525.50,118350.50,8,0.36
2,Total_Spending,0,0.00,68.75,1045.5,976.75,-1396.38,2510.62,3,0.13


In [4]:
# Lọc bản ghi bất hợp lý, chia 80% Train / 20% Test rồi mới điền trung vị Income của tập Train
valid_df = add_derived_features(filter_invalid_records(raw_df))
train_raw, test_raw = split_data(valid_df, save_files=False)
train_medians = compute_train_medians(train_raw)

train_df = clean_data(train_raw, fill_medians=train_medians, drop_invalid_outliers=False)
test_df = clean_data(test_raw, fill_medians=train_medians, drop_invalid_outliers=False)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)
pd.concat([train_df, test_df], ignore_index=True).to_csv(
    PROCESSED_DATA_DIR / "full_customers.csv", index=False
)

print("Thống kê mô tả trên tập Train (80%):")
train_df[NUMERICAL_FEATURES].describe().T

 [LỌC BẢN GHI BẤT HỢP LÝ] Đã loại 4 dòng nhiễu cực đoan/trùng lặp.
 [CHIA TẬP 80/20] Tổng: 2,236 -> Train: 1,788 (80%) | Test: 448 (20%)
 [XỬ LÝ KHUYẾT] Cột 'Income': đã điền 18 giá trị thiếu bằng trung vị (51,650.50).
 [XỬ LÝ KHUYẾT] Cột 'Income': đã điền 6 giá trị thiếu bằng trung vị (51,650.50).
Thống kê mô tả trên tập Train (80%):


,count,mean,std,min,25%,50%,75%,max
Age,1788.0,45.229866,11.853908,18.0,37.0,44.0,55.0,74.0
Income,1788.0,51975.558725,21567.016696,2447.0,35196.0,51650.5,68179.5,162397.0
Total_Spending,1788.0,605.598993,598.817450,5.0,69.0,397.5,1043.5,2524.0


In [5]:
# Trực quan hóa phân phối, Boxplot phát hiện ngoại lai và tương quan song biến trên tập Train
plot_eda_distributions(train_df)
plot_eda_correlations(train_df)

 [ĐÃ LƯU BIỂU ĐỒ] -> results\figures\00a_eda_distributions.png
 [ĐÃ LƯU BIỂU ĐỒ] -> results\figures\00b_eda_correlations.png


WindowsPath('E:/Hoc_May_Tren_truong/lab_3.1_CUSTOMER_CLUSTERING/results/figures/00b_eda_correlations.png')